# 🌱 Farm Monitor

This notebook reads sensor data and produces:
- **Temperature analysis** — daily min/max and Growing Degree Days (GDD)
- **Soil moisture %** — converted from raw sensor readings using calibration
- **Smart watering log** — announces when the pump turns ON and OFF, and why

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime

# ── Load sensor data ───────────────────────────────────────
df = pd.read_csv('sensor_readings.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Loaded {len(df)} sensor readings")
print(f"Period : {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
print()
print(df.head(5).to_string(index=False))

---
## Part 1 — Temperature & Growing Degree Days (GDD)

**GDD formula:**
```
GDD per day = max(0,  ((T_max + T_min) / 2) - T_base)
```
We use **T_base = 10 °C** (standard baseline for many crops).

In [ ]:
T_BASE = 10.0   # °C baseline temperature

df['date'] = df['timestamp'].dt.date

# Daily min / max temperature
daily_temp = df.groupby('date')['temperature_c'].agg(
    t_min='min', t_max='max'
).reset_index()

daily_temp['t_avg']      = (daily_temp['t_max'] + daily_temp['t_min']) / 2
daily_temp['gdd_daily']  = (daily_temp['t_avg'] - T_BASE).clip(lower=0)
daily_temp['gdd_cumsum'] = daily_temp['gdd_daily'].cumsum()

print("Daily temperature summary and GDD:")
print(daily_temp.to_string(index=False, float_format=lambda x: f"{x:.2f}"))
print(f"\nTotal cumulative GDD over period: {daily_temp['gdd_cumsum'].iloc[-1]:.2f} °C·days")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle('Temperature Analysis – March 2026', fontsize=14, fontweight='bold', y=1.01)

# Panel 1: hourly temperature
ax1 = axes[0]
ax1.plot(df['timestamp'], df['temperature_c'], color='tomato', linewidth=1.4, label='Temperature (°C)')
ax1.fill_between(df['timestamp'], df['temperature_c'], alpha=0.12, color='tomato')
ax1.set_ylabel('Temperature (°C)', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, linestyle='--', alpha=0.35)
ax1.set_title('Hourly Temperature', fontsize=11)

# Panel 2: cumulative GDD
ax2 = axes[1]
dates = pd.to_datetime(daily_temp['date'])
ax2.bar(dates, daily_temp['gdd_daily'], color='steelblue', alpha=0.6, label='GDD per day', width=0.4)
ax2.plot(dates, daily_temp['gdd_cumsum'], color='navy', linewidth=2,
         marker='o', markersize=5, label='Cumulative GDD')
ax2.set_ylabel('GDD (°C·days)', fontsize=11)
ax2.set_xlabel('Date', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, linestyle='--', alpha=0.35)
ax2.set_title('Growing Degree Days', fontsize=11)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.savefig('temperature_gdd.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: temperature_gdd.png")

---
## Part 2 — Soil Moisture %

We convert raw sensor readings (0–1023) to gravimetric soil moisture % using our calibration:

```
Soil Moisture % = ((W_wet - W_dry) / W_dry) × 100
```

The calibration samples gave us a best-fit linear equation:
- **slope** = –0.0220  
- **intercept** = 19.87

So: `moisture_pct = slope × sensor_reading + intercept`

In [ ]:
# Calibration parameters (from soil moisture calibration notebook)
# Derived from 5 physical samples: readings 110–823 mapped to moisture 7.9%–16.6%
SLOPE     = -0.0220
INTERCEPT =  19.87

# Convert raw sensor reading to soil moisture %
df['soil_moisture_pct'] = (SLOPE * df['soil_sensor_raw'] + INTERCEPT).round(2)

# Clamp to realistic range
df['soil_moisture_pct'] = df['soil_moisture_pct'].clip(lower=0, upper=40)

print("Sample of converted soil moisture readings:")
print(df[['timestamp', 'soil_sensor_raw', 'soil_moisture_pct']].head(10).to_string(index=False))
print(f"\nSoil moisture range: {df['soil_moisture_pct'].min():.2f}% – {df['soil_moisture_pct'].max():.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(df['timestamp'], df['soil_moisture_pct'], color='saddlebrown',
        linewidth=1.4, label='Soil Moisture %')
ax.fill_between(df['timestamp'], df['soil_moisture_pct'], alpha=0.15, color='saddlebrown')

# Threshold line
THRESHOLD_RAW = 580
THRESHOLD_PCT = round(SLOPE * THRESHOLD_RAW + INTERCEPT, 2)
ax.axhline(THRESHOLD_PCT, color='red', linestyle='--', linewidth=1.2,
           label=f'Watering threshold ({THRESHOLD_PCT}%)')

ax.set_ylabel('Soil Moisture (%)', fontsize=11)
ax.set_xlabel('Date', fontsize=11)
ax.set_title('Soil Moisture Over Time – March 2026', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d %H:%M'))
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('soil_moisture.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: soil_moisture.png")

---
## Part 3 — Smart Watering Log

Logic:
- If soil moisture % drops **below the threshold** → pump turns **ON**
- Pump runs for exactly the time needed to reach the target moisture
- Pump turns **OFF**, event is logged

Rate: **0.446% moisture per second** of pumping (calibrated from 20.3 raw units/sec)

In [ ]:
# Watering settings (converted to % units)
THRESHOLD_PCT  = round(SLOPE * 580 + INTERCEPT, 2)   # below this → water
TARGET_PCT     = round(SLOPE * 500 + INTERCEPT, 2)   # water until we reach this
RATE_PCT_SEC   = abs(SLOPE * 20.3)                   # % drop per second of pumping

print(f"Watering threshold : {THRESHOLD_PCT:.2f}% soil moisture")
print(f"Target moisture    : {TARGET_PCT:.2f}% soil moisture")
print(f"Pump rate          : {RATE_PCT_SEC:.4f}% per second")
print()

watering_log = []
pump_is_on   = False
total_pump_s = 0.0

for _, row in df.iterrows():
    ts      = row['timestamp']
    moist   = row['soil_moisture_pct']
    raw     = row['soil_sensor_raw']

    if moist < THRESHOLD_PCT and not pump_is_on:
        # Need to water
        needed_pct = TARGET_PCT - moist
        pump_secs  = max(1.0, min(needed_pct / RATE_PCT_SEC, 30.0))
        total_pump_s += pump_secs
        pump_is_on = True

        msg_on  = (f"[{ts}]  💧 PUMP ON   "
                   f"moisture={moist:.2f}% (below {THRESHOLD_PCT:.2f}%)  "
                   f"→ running for {pump_secs:.1f}s to reach {TARGET_PCT:.2f}%")
        msg_off = (f"[{ts}]  ⏹  PUMP OFF  "
                   f"ran {pump_secs:.1f}s | total pump time so far: {total_pump_s:.1f}s")

        print(msg_on)
        print(msg_off)
        print()

        watering_log.append({
            'timestamp'       : str(ts),
            'soil_moisture_pct': moist,
            'raw_reading'     : int(raw),
            'event'           : 'PUMP ON',
            'pump_duration_s' : round(pump_secs, 2),
            'total_pump_s'    : round(total_pump_s, 2),
            'note'            : f'watering to reach {TARGET_PCT:.2f}%'
        })
        watering_log.append({
            'timestamp'       : str(ts),
            'soil_moisture_pct': moist,
            'raw_reading'     : int(raw),
            'event'           : 'PUMP OFF',
            'pump_duration_s' : round(pump_secs, 2),
            'total_pump_s'    : round(total_pump_s, 2),
            'note'            : f'completed'
        })
        pump_is_on = False

    else:
        watering_log.append({
            'timestamp'       : str(ts),
            'soil_moisture_pct': moist,
            'raw_reading'     : int(raw),
            'event'           : '',
            'pump_duration_s' : 0,
            'total_pump_s'    : round(total_pump_s, 2),
            'note'            : 'no watering needed'
        })

log_df = pd.DataFrame(watering_log)
log_df.to_csv('watering_log.csv', index=False)
print(f"Total pump events  : {len(log_df[log_df['event']=='PUMP ON'])}")
print(f"Total pump time    : {total_pump_s:.1f} seconds")
print(f"Log saved          : watering_log.csv")

---
## Part 4 — Combined Dashboard

In [ ]:
pump_events = log_df[log_df['event'] == 'PUMP ON'].copy()
pump_events['timestamp'] = pd.to_datetime(pump_events['timestamp'])

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Farm Monitor Dashboard – March 2026', fontsize=15, fontweight='bold', y=1.01)

# Panel 1: Temperature
ax1 = axes[0]
ax1.plot(df['timestamp'], df['temperature_c'], color='tomato', linewidth=1.3)
ax1.fill_between(df['timestamp'], df['temperature_c'], alpha=0.1, color='tomato')
ax1.set_ylabel('Temp (°C)', fontsize=10)
ax1.set_title('Temperature', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.3)

# Panel 2: Soil Moisture %
ax2 = axes[1]
ax2.plot(df['timestamp'], df['soil_moisture_pct'], color='saddlebrown', linewidth=1.3)
ax2.fill_between(df['timestamp'], df['soil_moisture_pct'], alpha=0.12, color='saddlebrown')
ax2.axhline(THRESHOLD_PCT, color='red', linestyle='--', linewidth=1,
            label=f'Threshold ({THRESHOLD_PCT:.1f}%)')
ax2.axhline(TARGET_PCT, color='green', linestyle=':', linewidth=1,
            label=f'Target ({TARGET_PCT:.1f}%)')

# Mark pump events on soil chart
for _, pe in pump_events.iterrows():
    ax2.axvline(pe['timestamp'], color='royalblue', linewidth=1.2, alpha=0.6)

ax2.set_ylabel('Soil Moisture (%)', fontsize=10)
ax2.set_title('Soil Moisture with Watering Events (blue lines = pump ON)', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(True, linestyle='--', alpha=0.3)

# Panel 3: Cumulative GDD
ax3 = axes[2]
dates_plot = pd.to_datetime(daily_temp['date'])
ax3.step(dates_plot, daily_temp['gdd_cumsum'], color='navy', linewidth=2, where='post')
ax3.fill_between(dates_plot, daily_temp['gdd_cumsum'], alpha=0.15, color='navy', step='post')
ax3.set_ylabel('Cumulative GDD', fontsize=10)
ax3.set_xlabel('Date', fontsize=10)
ax3.set_title('Cumulative Growing Degree Days', fontsize=11)
ax3.grid(True, linestyle='--', alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('farm_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: farm_dashboard.png")

---
## Summary

| Metric | Value |
|--------|-------|
| Period covered | 2026-03-01 → 2026-03-03 |
| Total hourly readings | 64 |
| Temperature range | printed above |
| Cumulative GDD | printed above |
| Soil moisture range | printed above |
| Watering threshold | printed above |
| Total pump events | printed above |
| Total pump time | printed above |

All results are expressed in **percentage units (%)** for soil moisture and **°C·days** for GDD.